# Olist Dataset - Initial Exploration

just getting familiar with the data before writing any pipeline code  
9 csv files from kaggle - brazilian ecommerce data

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# load all 9 files
DATA = "../data/"

customers = pd.read_csv(DATA + "olist_customers_dataset.csv")
orders = pd.read_csv(DATA + "olist_orders_dataset.csv")
order_items = pd.read_csv(DATA + "olist_order_items_dataset.csv")
order_payments = pd.read_csv(DATA + "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(DATA + "olist_order_reviews_dataset.csv")
products = pd.read_csv(DATA + "olist_products_dataset.csv")
sellers = pd.read_csv(DATA + "olist_sellers_dataset.csv")
geolocation = pd.read_csv(DATA + "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(DATA + "product_category_name_translation.csv")

print("done")

In [ ]:
# check sizes
dfs = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "category_translation": category_translation
}

for name, df in dfs.items():
    print(f"{name}: {df.shape}")

geolocation has like 1M rows lol, thats a lot  
orders is ~100k which makes sense  
category_translation is tiny - just a mapping table

In [ ]:
# look at columns
for name, df in dfs.items():
    print(f"\n--- {name} ---")
    print(df.columns.tolist())

In [ ]:
orders.head()

In [ ]:
order_items.head()

ok so order_items has price and freight_value - thats what ill use for revenue  
orders has all the timestamps - purchase, approved, delivered etc

In [ ]:
# check nulls in each df
for name, df in dfs.items():
    nulls = df.isnull().sum()
    if nulls.any():
        print(f"\n--- {name} ---")
        print(nulls[nulls > 0])

## null observations

- orders: `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date` have nulls - makes sense, some orders not delivered yet
- order_reviews: `review_comment_title` and `review_comment_message` have a TON of nulls - people dont always leave comments
- products: `product_category_name` has some nulls, also weight/dimensions columns - need to handle this

products nulls are annoying, will need to deal with those in transform

In [ ]:
orders.info()

all the date columns are object type rn, need to convert to datetime - note to self

In [ ]:
orders.describe()

In [ ]:
order_items.describe()

price goes up to 6735 - there are some expensive items  
freight_value max is 409, mean is around 19  
some order_items have price 0?? need to check that

In [ ]:
# order status distribution - interesting to see
orders['order_status'].value_counts()

most orders are delivered, a few cancelled/unavailable  
theres also 'invoiced' and 'processing' - probably live orders at time of dataset export

In [ ]:
# how many unique customers vs orders
print("unique customers:", customers['customer_unique_id'].nunique())
print("total customer rows:", len(customers))
print("total orders:", len(orders))

interesting - customers table has a `customer_id` (per order) and `customer_unique_id` (actual person)  
so same person can appear multiple times with different customer_id  
need to use customer_unique_id for repeat customer analysis

In [ ]:
# payment types
order_payments['payment_type'].value_counts()

In [ ]:
# quick look at review scores
order_reviews['review_score'].value_counts().sort_index()

## stuff to handle in transform.py

- convert date columns from string to datetime
- products table has nulls in category name - fill or drop
- need to join category_translation to get english names
- customer_unique_id vs customer_id thing
- filter out cancelled orders probably
- price = 0 in order_items - check if real or data issue

TODO: figure out how to calculate delivery time (delivered_customer_date - purchase_timestamp)